In [1]:
import pandas as pd
import json
import random

# 1. LOAD THE CSV FILE
print("📖 Reading CSV file... (this might take 5-10 seconds)")
df = pd.read_csv('medicine_dataset.csv')
print(f"✅ Loaded! Total records: {len(df)}")

📖 Reading CSV file... (this might take 5-10 seconds)


C:\Users\vicky\AppData\Local\Temp\ipykernel_12812\1041519798.py:7: DtypeWarning: Columns (42,43,44,45,46,47,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('medicine_dataset.csv')


✅ Loaded! Total records: 248218


In [ ]:
# 2. INSPECT COLUMNS AND FIND ACTUAL COLUMN NAMES
print('\n📋 Columns:')
print(df.columns.tolist())
print('\n📊 Dtypes:')
print(df.dtypes)

def normalize_column_name(value):
    return ''.join(ch.lower() for ch in str(value) if ch.isalnum())

column_lookup = {normalize_column_name(col): col for col in df.columns}
def find_column(*candidates):
    for candidate in candidates:
        match = column_lookup.get(normalize_column_name(candidate))
        if match is not None:
            return match
    return None

name_col = find_column('name')
therapeutic_col = find_column('Therapeutic_Class', 'therapeutic class')
discontinued_col = find_column('Is_discontinued', 'is discontinued')
if name_col is None or therapeutic_col is None:
    raise KeyError(f'Required name or therapeutic class column missing. Found: {df.columns.tolist()}')

def numbered_columns(prefix, max_number):
    return [col for n in range(max_number + 1) if (col := find_column(f'{prefix}{n}')) is not None]

side_effect_cols = numbered_columns('sideEffect', 40)
use_cols = numbered_columns('use', 4)
substitute_cols = numbered_columns('substitute', 4)
price_col = find_column('price(₹)', 'price', 'mrp', 'price in rupees')
if price_col is None:
    price_col = next((c for c in df.columns if 'price' in normalize_column_name(c)), None)

# 3. FILTER FOR EVERYDAY MEDICINES; missing optional discontinued column means no exclusion is possible.
everyday_terms = ['analgesic', 'anti-inflammatory', 'antacid', 'antihistamine', 'cough', 'cold', 'fever', 'vitamin', 'mineral', 'supplement', 'dermatological', 'topical', 'dental', 'ophthalmic', 'antiulcer', 'laxative', 'antidiarrheal', 'antibiotic', 'antiseptic', 'expectorant', 'decongestant', 'throat', 'skin', 'antifungal', 'respiratory', 'antiemetic', 'gastrointestinal']
name_terms = ['tablet', 'capsule', 'syrup', 'drops', 'ointment', 'gel', 'cream', 'spray', 'suspension']
excluded_terms = ['oncology', 'cardiac emergency', 'anesthesia', 'anaesthesia', 'psychiatric', 'chemotherapy', 'transplant', 'intensive care']
therapeutic_text = df[therapeutic_col].fillna('').astype(str).str.lower()
name_text = df[name_col].fillna('').astype(str).str.lower()
is_everyday = therapeutic_text.str.contains('|'.join(everyday_terms), regex=True, na=False)
has_medicine_form = name_text.str.contains('|'.join(name_terms), regex=True, na=False)
is_not_severe = ~therapeutic_text.str.contains('|'.join(excluded_terms), regex=True, na=False)
if discontinued_col is None:
    is_not_discontinued = pd.Series(True, index=df.index)
else:
    discontinued_text = df[discontinued_col].fillna('').astype(str).str.strip().str.lower()
    is_not_discontinued = discontinued_text.isin(['false', '0'])
filtered = df.loc[is_everyday & has_medicine_form & is_not_severe & is_not_discontinued].copy()
print(f'\n✅ Rows remaining after filtering: {len(filtered):,}')

# 4. REPRODUCIBLY SHUFFLE AND TAKE UP TO 4,500 ROWS
sampled = filtered.sample(n=min(4500, len(filtered)), random_state=42).copy()
print(f'🎯 Final sample count: {len(sampled):,}')

# 5. CLEAN WIDE-COLUMN VALUES INTO LISTS
def clean_list_value(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return None if not text or text.lower() == 'nan' else text

def collect_values(row, columns):
    return [value for col in columns if (value := clean_list_value(row[col])) is not None]

# 6. MAP SAMPLED ROWS TO THE CLEAN JSON SHAPE
def get_string(row, column):
    if column is None or pd.isna(row[column]):
        return ''
    value = str(row[column]).strip()
    return '' if value.lower() == 'nan' else value

def is_habit_forming(value):
    return False if pd.isna(value) else str(value).strip().lower() in {'yes', 'true'}

composition1_col = find_column('short_composition1')
composition2_col = find_column('short_composition2')
manufacturer_col = find_column('manufacturer_name', 'manufacturer')
action_col = find_column('Action_Class', 'action class')
chemical_col = find_column('Chemical_Class', 'chemical class')
habit_col = find_column('Habit_Forming', 'habit forming')
records = []
for _, row in sampled.iterrows():
    records.append({
        'name': get_string(row, name_col),
        'composition1': get_string(row, composition1_col),
        'composition2': get_string(row, composition2_col),
        'price': get_string(row, price_col),
        'manufacturer': get_string(row, manufacturer_col),
        'isDiscontinued': False,
        'sideEffects': collect_values(row, side_effect_cols),
        'uses': collect_values(row, use_cols),
        'substitutes': collect_values(row, substitute_cols),
        'therapeuticClass': get_string(row, therapeutic_col),
        'actionClass': get_string(row, action_col),
        'chemicalClass': get_string(row, chemical_col),
        'habitForming': is_habit_forming(row[habit_col]) if habit_col is not None else False,
    })

# 7. SAVE UTF-8 JSON AND REPORT THE RESULT
from pathlib import Path
output_path = Path('server/data/medicines_cleaned.json')
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open('w', encoding='utf-8') as output_file:
    json.dump(records, output_file, ensure_ascii=False, indent=2)
file_size_mb = output_path.stat().st_size / (1024 * 1024)
print(f'\n💾 Saved {len(records):,} records to {output_path}')
print(f'📦 File size: {file_size_mb:.2f} MB')
print('\n🔎 Sample record:')
print(json.dumps(records[0] if records else None, ensure_ascii=False, indent=2))